# THEREDA → GEMS3 Export

Exports the ThermoHub-schema graph produced by `import-THEREDA-json-data.ipynb` into GEMS3 backup
format (`IComp`/`DComp`/`ReacDC`/`Phase`/`SDref` JSON) via `ExportToGems3`. Run this after the
import notebook has populated the target database.

Equivalent standalone script: `export-gems3.py` (same three steps, runnable headlessly via
`python export-gems3.py` — useful for testing a fix without touching a live Jupyter kernel, since a
running kernel can't hot-reload a rebuilt `thermomatch` extension).

In [1]:
import thermomatch as match
match.ThermoImpexGenerator.create_local_database_if_not_existent = True

Same `thermomatch` import as the import notebook. No `create_local_database_if_not_existent` flag
here — export assumes the target database already exists and was populated by the import
notebook.

In [2]:
# Set path to `schemas` and `lua` not from the configuration file
match.ThermoImpexGenerator.setResourcesDirectory("../../Resources")

# THEREDA-specific redox/formula/symbol-shortening/SDref-author rules - used at both import time
# (formula charge-fixing) and export time (symbol shortening, SDref author keys); process-wide,
# so load it before both runImport() and ExportAllJson() below.
match.loadGems3ExportRulesFile("scripts-out/gems3-export-rules.json")

[2026-09-03 10:50:07.140] [jsonio17] [info] Home directory is /home/dmiron
[thermomatch] [info] jsonio17::ioSettings() is reading settings from "../../Resources/ThermoMatch-config.json" (resolved to "/home/dmiron/git/hub/thermoimpex-jupyter/databases/THEREDA/../../Resources/ThermoMatch-config.json", cwd "/home/dmiron/git/hub/thermoimpex-jupyter/databases/THEREDA")
[thermomatch] [info] jsonio17::ioSettings() common.ResourcesDirectory = "../../Resources"
[thermomatch] [info] Loaded Gems3Export rules from "scripts-out/gems3-export-rules.json" (RedoxElements=29 FormulaOverrides=1 SymbolShortenRules=15 SDrefAuthorKeyOverrides=3 ModeCodeRules=7)


`loadGems3ExportRulesFile(...)` loads THEREDA's redox-element list, formula overrides, GEMS3
symbol-shortening regex rules, SDref author-key overrides, and `TPcMod`/`REcMod` mode-code override
rules — all as **process-wide** state (not tied to any one `ExportToGems3` instance), consumed by
`checkAndFixChargeFormula`, `shortenDCSymbol`, `ExportSDrefJson`, and `applyModeCodeRules`
respectively.

**Must be loaded before `ExportToGems3(...)` is constructed below** (and before any export calls
run) — there is no hardcoded C++ default any more, so skipping this call means zero redox-capable
elements, formula overrides, or symbol-shortening rules get applied during export.

In [3]:
# Records with properties.enabled == false (THEREDA "Declaration.enabled": not to be used in a
# geochemical code) are skipped by default. Set to False to export them too. Must be set BEFORE
# ExportToGems3(...) is constructed - the selection is made in the constructor.
match.ExportToGems3.skip_disabled = True

# Thermodynamic consistency check of the active reaction-defined substances (DFG/DFH/S/CP against the
# reaction values and the reactants, logK against DRG298, missing G data) - runs while the records are
# selected; one warning per mismatch in the log and in the text report below.
match.ExportToGems3.check_consistency = True
match.ExportToGems3.consistency_report_file = "data-out/consistency/thereda_export_consistency_report.txt"
# What is exported for an inconsistent record (ThermoHub itself is never changed):
#   "warn-only"   - the values as stored (default)
#   "prefer-logK" - keep logK298 / DRG298 / DRH / DRS / DRCP298 and export the formation values
#                   (DFG/DFH/S/CP298 in the ReacDC record) as implied by them and the reactants
match.ExportToGems3.consistency_mode = "warn-only"

# A reaction-defined substance without a formation Gibbs energy (DFG298):
#   derive_missing_g - derive it from the reaction (DRG298 or logK298) and the reactants' DFG298
#   skip_without_g   - if it still has no G0, do not export it: not as ReacDC, not in the phase species
#                      lists, and no pure phase left without species (listed in the report)
match.ExportToGems3.derive_missing_g = True
match.ExportToGems3.skip_without_g = True

#Export to GEMS3k format
gem_export = match.ExportToGems3("http://localhost:8529", "root", "", "ORD_THEREDA_R17", "thermodatasets/THEREDA2026;1:TDS_LMA;0");

[jsonarango] [info] You are connected to: arango 3.9.12
[jsonio17] [info] VertexSubstance loading collection: 604, loading query: 87264
[jsonio17] [info] VertexPhase loading collection: 943, loading query: 68667
[jsonio17] [info] VertexInteraction loading collection: 2493, loading query: 153847
[jsonio17] [info] VertexDataSource loading collection: 825, loading query: 20494
[thermomatch] [warning] Substance "(KCl)3(PbCl2)3H2O(s)": symbol could not be shortened to fit the GEMS3 16-character DC symbol limit (best effort: "(KCl)3(PbCl2)3w(s)", 18 characters); it will be truncated and disambiguated with a numeric suffix. Consider adding a "tag" for this substance, or a rule to the loaded Gems3Export rules file's "SymbolShortenRules".
[thermomatch] [warning] Substance "(MgCl2)3(PbCl2)19H2O(s)": symbol could not be shortened to fit the GEMS3 16-character DC symbol limit (best effort: "(MgCl2)3PbCl2w19(s)", 19 characters); it will be truncated and disambiguated with a numeric suffix. Consider

The 5-arg `ExportToGems3` constructor's last parameter is overloaded: a bare `idThermoDataSet`
string (as used here, `"thermodatasets/THEREDA2026;1:TDS_LMA;0"`) selects everything linked to that
thermodataset via `select_linked()`. Passing the same string wrapped in a Python list instead
silently binds to a *different* overload (`reaction_keys: vector<string>`) and produces an empty
selection with no error — keep this a plain string, not `[idThermoDataSet]`.

`ExportToGems3.skip_disabled` (class-level, default `True`) leaves out every substance, reaction, phase
and interaction whose `properties.enabled` is `false` (imported from THEREDA's `Declaration.enabled`),
and removes disabled species from the phase species lists. Set it before constructing the exporter.

In [4]:
import os

# Output sub-folder of data-out. Pass a bare name (no "/") to ExportAllJson and chdir into data-out
# instead: a path containing "/" makes it tag the files after the last path segment
# (IComp.10_2026.json) instead of the expected IComp.backup.json.
OUT_DIR = "10_2026"
os.makedirs(f"data-out/{OUT_DIR}", exist_ok=True)
os.chdir("data-out")
try:
    gem_export.ExportAllJson(OUT_DIR);
finally:
    os.chdir("..")

# Records of the cement phases listed in the map (also in CEMDATA) go to separate sets of files,
# THEREDA.cemdata.<version>; the main files are rewritten without them. Set SPLIT_MAP = None to skip.
import sys
sys.path.append("../..")
from gems3_split import split_gems_records

SPLIT_MAP = "scripts-out/gems3-split-map.json"
split_groups = split_gems_records(SPLIT_MAP, f"data-out/{OUT_DIR}") if SPLIT_MAP else []

# the files with the pure phases (every pure phase, also those already in the Phase files) go to a subfolder
import glob
os.makedirs(f"data-out/{OUT_DIR}/pure", exist_ok=True)
for path in glob.glob(f"data-out/{OUT_DIR}/*pure*.json"):
    os.replace(path, os.path.join(f"data-out/{OUT_DIR}/pure", os.path.basename(path)))

[jsonio17] [info] VertexElement loading collection: 46, loading query: 3894
[jsonio17] [info] VertexSubstance loading collection: 42, loading query: 87886
[thermomatch] [info] Formula "Np+3" had a charge imbalance; fixed by setting Np valence to 3: "Np|+3|+3"
[thermomatch] [info] Formula "Np+4" had a charge imbalance; fixed by setting Np valence to 4: "Np|+4|+4"
[thermomatch] [info] Formula "NpO2+" had a charge imbalance; fixed by setting Np valence to 5: "Np|+5|O2+"
[thermomatch] [info] Formula "Pu+3" had a charge imbalance; fixed by setting Pu valence to 3: "Pu|+3|+3"
[thermomatch] [info] Formula "Pu+4" had a charge imbalance; fixed by setting Pu valence to 4: "Pu|+4|+4"
[thermomatch] [info] Formula "PuO2+" had a charge imbalance; fixed by setting Pu valence to 5: "Pu|+5|O2+"
[thermomatch] [info] Formula "SeO4-2" had a charge imbalance; fixed by setting Se valence to 6: "Se|+6|O4-2"
[thermomatch] [info] Formula "TcO(OH)2@" had a charge imbalance; fixed by setting Tc valence to 4: "Tc

`gems3_split.py` (map: `scripts-out/gems3-split-map.json`) splits the exported records into several sets, applied in the order of the map:
1. `cemdata` – solid species with the element composition of a CEMDATA 18 species, with their phases (pure phases and solid solutions with end-members, only when all species match) → `*.THEREDA.cemdata.<version>`;
2. `aqueous`, `solids`, `gases` – what is left, by phase state (`a`; `s`; `g`/`f`) → `*.THEREDA.main.aqueous.<version>`, `.main.solids`, `.main.gases`. The gas set has species only (the gas phase is not exported).

The elements (IComp) stay in the main set, `icomp.THEREDA.<version>`, and ALL data sources in the single `sdref.THEREDA.<version>` (a data-source key may exist in one database of a GEMS project only, so they are not split); empty main files are not converted. The log lists what was moved and warns about map entries that match nothing, species without a record, phases with only some species in a group, and species still used by a phase outside it.

Writes `IComp`/`DComp`/`ReacDC`/`SDref` JSON into `data-out/<OUT_DIR>` (`10_2026`, so the result can be compared with the older files directly in `data-out`), plus phases split across two files:
`Phase.<name>.json` (solid-solution/mixing-model phases, plus any pure phase whose substance isn't
part of a solid solution) and `Phase.<name>.pure.json` (every `mixmod==0` pure phase, including
ones already covered by the mixed file). If `log.module_level.thermomatch` is set to `"debug"` in
`ThermoMatch-config.json`, this call also dumps the full per-reaction JSON to the log for every
reaction — harmless for a plain script, but can overwhelm Jupyter's output rendering.

Known post-export caveat noted below: GEMS3 flips the sign of the second Redlich-Kister parameter
relative to how it's stored here, so exported `RK_...` phase files may need that sign corrected by
hand before use in GEMS3.

In [5]:
# in GEMS second RK parameters changes sign RK_(SO4)Ca2Al0.6	1	0	1.67	-0.946 --> RK_(SO4)Ca2Al0.6	1	0	1.67	0.946

o the loaded Gems3Export rules file's "SymbolShortenRules".
[thermomatch] [warning] Phase "CdCl2(Cd(OH)2)3(cr)": "symbol" could not be shortened to fit the GEMS3 16-character phase name limit (best effort: "CdCl2(Cd(OH)2)3(cr)", 19 characters); it will be truncated and disambiguated with a numeric suffix. Consider adding a rule to the loaded Gems3Export rules file's "SymbolShortenRules".
[thermomatch] [warning] Phase "CdCl2(Cd(OH)2)3(cr)": "name" could not be shortened to fit the GEMS3 16-character phase name limit (best effort: "CdCl2(Cd(OH)2)3(cr)", 19 characters); it will be truncated and disambiguated with a numeric suffix. Consider adding a rule to the loaded Gems3Export rules file's "SymbolShortenRules".
[thermomatch] [warning] Phase "CdCl2(MgCl2)212H2O(cr)": "symbol" could not be shortened to fit the GEMS3 16-character phase name limit (best effort: "CdCl2(MgCl2)2w12(cr)", 20 characters); it will be truncated and disambiguated with a numeric suffix. Consider adding a rule to the

## Also build GEM-Selektor `.pdb`/`.ndx` files (via `json2db`)

`ExportAllJson(...)` above only writes the intermediate `*.backup.json` files. This step converts
each of them into a real GEM-Selektor `<keyword>.<tag>.pdb`/`.ndx` pair in `data-out/gems-auto`,
using GEMSGUI's headless `json2db` CLI tool (see `gemsgui_import.py` at the repo root, and
GEMSGUI's own `CLAUDE.md`, "benchcomp/json2db" section, for how that tool works). Requires
`json2db` to be installed (`bash conda-install-dependencies.sh`, run once per environment).

The two `Phase.backup*.json` files both map to the `phase` keyword but overlap on some keys (per
the note above, `Phase.backup.pure.json` re-includes every pure phase also present in
`Phase.backup.json`) — writing them into the same `.pdb` would silently let the pure-phase version
win on any overlapping key, so they get distinct tags (`...` and `....pure`) instead.

In [6]:
import json
import sys
sys.path.append("../..")
from gemsgui_import import json_to_db

THEREDA_VERSION = "ver2026-01-r17"


def tag(*parts):
    """THEREDA[.<set>][.pure].<version> - e.g. THEREDA.main.solids.pure.ver2026-01-r17"""
    return ".".join(["THEREDA", *[p for p in parts if p], THEREDA_VERSION])


for json_name, keyword, set_tag in [
    ("IComp.backup.json", "icomp", tag()),
    ("DComp.backup.json", "dcomp", tag()),
    ("Phase.backup.json", "phase", tag()),
    ("Phase.backup.pure.json", "phase", tag("pure")),
    ("ReacDC.backup.json", "reacdc", tag()),
    ("SDref.backup.json", "sdref", tag()),
]:
    # the main file may be empty after the split (everything went to the group sets): no empty database then
    sub = "pure/" if "pure" in json_name else ""
    if json.load(open(f"data-out/{OUT_DIR}/{sub}{json_name}")):
        json_to_db(f"data-out/{OUT_DIR}/{sub}{json_name}", keyword=keyword, output_dir=f"data-out/{OUT_DIR}/gems-auto/{sub}", tag=set_tag)

# the split-off sets (e.g. cement phases): THEREDA.cemdata.<version>
for group in split_groups:
    for json_name, keyword, suffix in group["files"]:
        sub = "pure/" if "pure" in json_name else ""
        json_to_db(f"data-out/{OUT_DIR}/{sub}{json_name}", keyword=keyword, output_dir=f"data-out/{OUT_DIR}/gems-auto/{sub}", tag=tag(group["tag_infix"], suffix.strip(".")))
print("done")

done


## Compositions (`compos`)

`data-in/Compos.backup.json` (hand-made, GEMS PCO recipes: 1 mol of a salt, mineral, gas or acid as amounts of elements) and `data-in/Compos.extra.json` (more salts, minerals and gases made from the elements of THEREDA by `make_compos_extra.py`; re-run it after changing its list) are merged – a key in `Compos.backup.json` wins – and written to `Compos.backup.json` in the output folder. All elements must exist in `IComp`. They are converted into `compos.THEREDA.<version>.pdb/.ndx` when the installed `json2db` knows the `compos` keyword.

In [ ]:
import json

icomp_symbols = {rec["key"][0] for rec in json.load(open(f"data-out/{OUT_DIR}/IComp.backup.json"))}
compos = json.load(open("data-in/Compos.backup.json"))
known = {tuple(rec["key"]) for rec in compos}
compos += [rec for rec in json.load(open("data-in/Compos.extra.json")) if tuple(rec["key"]) not in known]

for rec in compos:
    missing = [row[0][:6].strip() for row in rec["dod"]["symIC"] if row[0][:6].strip() not in icomp_symbols]
    if missing:
        print(f"WARNING: compos {rec['key'][0]} uses elements that are not in IComp: {missing}")

with open(f"data-out/{OUT_DIR}/Compos.backup.json", "w") as fh:
    json.dump(compos, fh, indent=4)
print(len(compos), "compositions")

try:
    json_to_db(f"data-out/{OUT_DIR}/Compos.backup.json", keyword="compos",
               output_dir=f"data-out/{OUT_DIR}/gems-auto", tag=tag())
except RuntimeError as err:
    print("compos .pdb/.ndx not written (json2db without the compos keyword?):", str(err)[:300])